In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window 

# Load Bronze
df = spark.read.table("discussion_forum_bronze")

# Define schema
schema = StructType([
    StructField("post_id", StringType()),
    StructField("thread_id", StringType()),
    StructField("course_id", StringType()),
    StructField("author_student_id", StringType()),
    StructField("parent_post_id", StringType()),
    StructField("post_ts", TimestampType()),
    StructField("content_length_chars", IntegerType()),
    StructField("has_attachment", BooleanType()),
    StructField("like_count", IntegerType()),
    StructField("reply_count", IntegerType()),
    StructField("is_instructor_post", BooleanType())
])

# =========================
# 1. Parse JSON
# =========================
parsed_df = df.withColumn("data", from_json(col("raw_json"), schema)).select("data.*")

# Basic data quality filter
parsed_df = parsed_df.filter(col("post_id").isNotNull())

# =========================
# 2. Compute post_depth (0,1,2)
# =========================

# Create parent lookup
parent_df = parsed_df.select(
    col("post_id").alias("p_post_id"),
    col("parent_post_id").alias("p_parent_id")
)

# Join to find parent of parent
parsed_df = parsed_df.join(
    parent_df,
    parsed_df.parent_post_id == parent_df.p_post_id,
    "left"
)

# Depth logic
parsed_df = parsed_df.withColumn(
    "post_depth",
    when(col("parent_post_id").isNull(), 0)              # top-level
    .when(col("p_parent_id").isNull(), 1)                # reply
    .otherwise(2)                                       # reply-to-reply
)

# =========================
# 3. Thread post count
# =========================

thread_window = Window.partitionBy("thread_id")

parsed_df = parsed_df.withColumn(
    "thread_post_count",
    count("*").over(thread_window)
)

# =========================
# 4. Instructor reply rate
# =========================

parsed_df = parsed_df.withColumn(
    "is_student_post",
    when(col("is_instructor_post") == False, 1).otherwise(0)
)

agg_thread = parsed_df.groupBy("thread_id").agg(
    sum(when(col("is_instructor_post"), 1).otherwise(0)).alias("instructor_posts"),
    sum("is_student_post").alias("student_posts")
)

# Safe division
agg_thread = agg_thread.withColumn(
    "instructor_reply_rate",
    when(col("student_posts") == 0, 0)
    .otherwise(col("instructor_posts") / col("student_posts"))
)

# Join back
final_df = parsed_df.join(agg_thread, "thread_id", "left")

# =========================
# 5. Write to Silver
# =========================

final_df.write.format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("edtech_project.edtech_silver.discussion_posts_parsed")



# =========================
# 6. Verification
# =========================

print("✅ Data written successfully to silver.discussion_posts_parsed")

# Sample output
display(final_df.limit(20))   # Databricks

# Row count
print("Total rows:", final_df.count())

# Null check
final_df.select(
    count(when(col("post_id").isNull(), True)).alias("null_post_id"),
    count(when(col("thread_id").isNull(), True)).alias("null_thread_id")
).show()